# Boosting Models — Demand Classification

**Models:** XGBoost · LightGBM · CatBoost  
**Target:** `demand_label` (binary: 0 = low demand, 1 = high demand)  
**Tuning:** Optuna (weighted-F1 objective)  

> **XGBoost note:** `use_label_encoder` has been removed — it was deprecated and removed in XGBoost ≥ 2.x.  
> **Class imbalance:** `scale_pos_weight = neg/pos` is set automatically for binary classification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score,
)

from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.style.use('dark_background')
SEED = 42

## 1. Load Data

In [ ]:
train_df = pd.read_csv('../data/splits/train.csv', low_memory=False)
test_df  = pd.read_csv('../data/splits/test.csv',  low_memory=False)

TARGET = 'demand_label'

DROP_COLS = [
    'demand_label', 'demand_label_3', 'demand_score',
    'Price_log', 'Price_original',
    'Price_vs_city_median',
]
raw_cols     = [c for c in train_df.columns if c.endswith('_raw')]
amenity_cols = [c for c in train_df.columns if 'Parsed Amenities' in c]
DROP_COLS   += raw_cols + amenity_cols

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

X_all  = train_df[feature_cols].fillna(0)
y_all  = train_df[TARGET]
X_test = test_df[feature_cols].fillna(0)
y_test = test_df[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
)

# Class imbalance weight (for binary classifiers)
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos if pos > 0 else 1.0

print(f'Train : {X_train.shape} | Val : {X_val.shape} | Test : {X_test.shape}')
print(f'scale_pos_weight = {scale_pos_weight:.3f}')

---
## 2. XGBoost

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators',   50,   500),
        'max_depth':       trial.suggest_int('max_depth',       3,    10),
        'learning_rate':   trial.suggest_float('learning_rate',0.01, 0.3),
        'subsample':       trial.suggest_float('subsample',    0.5,  1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree',0.5,1.0),
        'gamma':           trial.suggest_float('gamma',        0,    1),
        'reg_alpha':       trial.suggest_float('reg_alpha',    0,    1),
        'reg_lambda':      trial.suggest_float('reg_lambda',   0,    1),
        'scale_pos_weight': scale_pos_weight,
        'eval_metric':     'logloss',
        'random_state':    SEED,
    }
    model = XGBClassifier(**params, verbosity=0)
    model.fit(X_train, y_train)
    return f1_score(y_val, model.predict(X_val), average='weighted')

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=150, n_jobs=2)
print(f'Best Val F1 (XGB) : {study_xgb.best_value:.4f}')
print(f'Best Params       : {study_xgb.best_params}')

In [ ]:
X_combined = pd.concat([X_train, X_val])
y_combined = pd.concat([y_train, y_val])

xgb_model = XGBClassifier(
    **study_xgb.best_params,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=SEED,
    verbosity=0,
)
xgb_model.fit(X_combined, y_combined)
print('XGBoost trained.')

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

print('=== XGBoost ===')
print(f'Accuracy      : {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'F1 (weighted) : {f1_score(y_test, y_pred_xgb, average="weighted"):.4f}')
print(classification_report(y_test, y_pred_xgb, target_names=['Low','High']))

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_xgb), display_labels=['Low','High']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — XGBoost', fontsize=14); plt.tight_layout(); plt.show()

In [ ]:
lc_xgb = XGBClassifier(**study_xgb.best_params, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=SEED, verbosity=0)
ts, tr_s, val_s = learning_curve(lc_xgb, X_train, y_train, cv=5, scoring='f1_weighted', train_sizes=np.linspace(0.1,1.0,10), n_jobs=-1)
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(ts, tr_s.mean(1), 'o-', label='Train F1')
ax.fill_between(ts, tr_s.mean(1)-tr_s.std(1), tr_s.mean(1)+tr_s.std(1), alpha=0.2)
ax.plot(ts, val_s.mean(1), 's-', label='Val F1')
ax.fill_between(ts, val_s.mean(1)-val_s.std(1), val_s.mean(1)+val_s.std(1), alpha=0.2)
ax.set(xlabel='Training Size', ylabel='F1 (Weighted)', title='Learning Curve — XGBoost')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 3. LightGBM

In [ ]:
def objective_lgbm(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators',    100, 500),
        'max_depth':        trial.suggest_int('max_depth',        -1,  15),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01,0.3),
        'num_leaves':       trial.suggest_int('num_leaves',       20, 150),
        'subsample':        trial.suggest_float('subsample',      0.5,  1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree',0.5,1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha',      0,    1),
        'reg_lambda':       trial.suggest_float('reg_lambda',     0,    1),
        'scale_pos_weight': scale_pos_weight,
    }
    model = LGBMClassifier(**params, random_state=SEED, verbose=-1)
    model.fit(X_train, y_train)
    return f1_score(y_val, model.predict(X_val), average='weighted')

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=150, n_jobs=2)
print(f'Best Val F1 (LGBM) : {study_lgbm.best_value:.4f}')
print(f'Best Params        : {study_lgbm.best_params}')

In [ ]:
lgbm_model = LGBMClassifier(**study_lgbm.best_params, scale_pos_weight=scale_pos_weight, random_state=SEED, verbose=-1)
lgbm_model.fit(X_combined, y_combined)
print('LightGBM trained.')

In [ ]:
y_pred_lgbm = lgbm_model.predict(X_test)

print('=== LightGBM ===')
print(f'Accuracy      : {accuracy_score(y_test, y_pred_lgbm):.4f}')
print(f'F1 (weighted) : {f1_score(y_test, y_pred_lgbm, average="weighted"):.4f}')
print(classification_report(y_test, y_pred_lgbm, target_names=['Low','High']))

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lgbm), display_labels=['Low','High']).plot(ax=ax, colorbar=False, cmap='Purples')
ax.set_title('Confusion Matrix — LightGBM', fontsize=14); plt.tight_layout(); plt.show()

In [ ]:
lc_lgbm = LGBMClassifier(**study_lgbm.best_params, scale_pos_weight=scale_pos_weight, random_state=SEED, verbose=-1)
ts, tr_s, val_s = learning_curve(lc_lgbm, X_train, y_train, cv=5, scoring='f1_weighted', train_sizes=np.linspace(0.1,1.0,10), n_jobs=-1)
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(ts, tr_s.mean(1), 'o-', label='Train F1'); ax.fill_between(ts, tr_s.mean(1)-tr_s.std(1), tr_s.mean(1)+tr_s.std(1), alpha=0.2)
ax.plot(ts, val_s.mean(1), 's-', label='Val F1'); ax.fill_between(ts, val_s.mean(1)-val_s.std(1), val_s.mean(1)+val_s.std(1), alpha=0.2)
ax.set(xlabel='Training Size', ylabel='F1 (Weighted)', title='Learning Curve — LightGBM')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 4. CatBoost

In [ ]:
def objective_cat(trial):
    params = {
        'iterations':    trial.suggest_int('iterations',   200,  600),
        'depth':         trial.suggest_int('depth',          4,   10),
        'learning_rate': trial.suggest_float('learning_rate',0.01,0.3),
        'l2_leaf_reg':   trial.suggest_float('l2_leaf_reg',  1,   10),
        'verbose':       0,
        'random_seed':   SEED,
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train)
    return f1_score(y_val, model.predict(X_val), average='weighted')

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=100, n_jobs=2)
print(f'Best Val F1 (Cat) : {study_cat.best_value:.4f}')
print(f'Best Params       : {study_cat.best_params}')

In [ ]:
cat_model = CatBoostClassifier(**study_cat.best_params, verbose=0, random_seed=SEED)
cat_model.fit(X_combined, y_combined)
print('CatBoost trained.')

In [ ]:
y_pred_cat = cat_model.predict(X_test)

print('=== CatBoost ===')
print(f'Accuracy      : {accuracy_score(y_test, y_pred_cat):.4f}')
print(f'F1 (weighted) : {f1_score(y_test, y_pred_cat, average="weighted"):.4f}')
print(classification_report(y_test, y_pred_cat, target_names=['Low','High']))

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_cat), display_labels=['Low','High']).plot(ax=ax, colorbar=False, cmap='Oranges')
ax.set_title('Confusion Matrix — CatBoost', fontsize=14); plt.tight_layout(); plt.show()

In [ ]:
lc_cat = CatBoostClassifier(**study_cat.best_params, verbose=0, random_seed=SEED)
ts, tr_s, val_s = learning_curve(lc_cat, X_train, y_train, cv=5, scoring='f1_weighted', train_sizes=np.linspace(0.1,1.0,10), n_jobs=-1)
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(ts, tr_s.mean(1), 'o-', label='Train F1'); ax.fill_between(ts, tr_s.mean(1)-tr_s.std(1), tr_s.mean(1)+tr_s.std(1), alpha=0.2)
ax.plot(ts, val_s.mean(1), 's-', label='Val F1'); ax.fill_between(ts, val_s.mean(1)-val_s.std(1), val_s.mean(1)+val_s.std(1), alpha=0.2)
ax.set(xlabel='Training Size', ylabel='F1 (Weighted)', title='Learning Curve — CatBoost')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 5. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model':         ['XGBoost', 'LightGBM', 'CatBoost'],
    'Accuracy':      [accuracy_score(y_test, p) for p in [y_pred_xgb, y_pred_lgbm, y_pred_cat]],
    'F1 (weighted)': [f1_score(y_test, p, average='weighted') for p in [y_pred_xgb, y_pred_lgbm, y_pred_cat]],
}).set_index('Model')

print(results.round(4))

results.plot(kind='bar', figsize=(9,5), rot=0, colormap='plasma')
plt.title('Boosting Models — Test Set Performance')
plt.ylabel('Score'); plt.ylim(0.5, 1.0); plt.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()